# Data Cleaning and Preparation

This notebook prepares the raw solar, weather, and Kaggle datasets for forecasting and anomaly detection.

## Notebook sections
1. Forecast data from DKASC and NASA weather sources.
2. Merge Kaggle generation and weather data.
3. Create a labeled Kaggle anomaly dataset.

All cleaned outputs are saved in the `data/processed` directory.

## 1. Forecast data preparation

Load DKASC and NASA weather data, convert timestamps, align observations by hour, and save `forecast_data.csv`.

In [1]:
import pandas as pd

dkasc_path = r'F:\Abdelrahman1\ME\programming\SIC\Project3.1\data\raw\Alice_Springs_2025.csv'

df = pd.read_csv(dkasc_path, nrows=2)

print("Irradiance / GHI:")
for col in df.columns:
    if 'Irradiance' in col or 'GHI' in col or 'Radiation' in col:
        print(f"- {col}")

print("\nTemperature:")
for col in df.columns:
    if 'Temperature' in col or 'Temp' in col:
        print(f"- {col}")

Irradiance / GHI:
- 101_DKA_WeatherStation_Global_Horizontal_Radiation
- 101_DKA_WeatherStation_Diffuse_Horizontal_Radiation
- 101_DKA_WeatherStation_Radiation_Global_Tilted
- 101_DKA_WeatherStation_Radiation_Diffuse_Tilted

Temperature:
- 101_DKA_WeatherStation_Weather_Temperature_Celsius


In [8]:
import pandas as pd
import os

RAW_DIR = r'F:\Abdelrahman1\ME\programming\SIC\Project3.2\data\raw'
PROCESSED_DIR = r'F:\Abdelrahman1\ME\programming\SIC\Project3.2\data\processed\Forecast'
dkasc_path = os.path.join(RAW_DIR, 'Alice_Springs_2025.csv')

print("⏳ جاري قراءة DKASC...")
df = pd.read_csv(dkasc_path, on_bad_lines='skip', low_memory=False)

# 1. هنستخدم العمود اللي لاقيناه
pv_col = '241_DKA_Totals_PV_Active_Power'

# 2. تجهيز داتا الإنتاج
df_power = df[['timestamp', pv_col]].copy()
df_power.rename(columns={pv_col: 'actual_power_kw'}, inplace=True)
df_power['timestamp'] = pd.to_datetime(df_power['timestamp'], errors='coerce')
df_power['actual_power_kw'] = pd.to_numeric(df_power['actual_power_kw'], errors='coerce')

# 💡 السر هنا: بنحول الأرقام السالبة لصفر عشان الضوضاء
df_power['actual_power_kw'] = df_power['actual_power_kw'].clip(lower=0) 
df_power = df_power.dropna().sort_values('timestamp')

# 3. تجهيز داتا الطقس
weather_cols = [
    '101_DKA_WeatherStation_Global_Horizontal_Radiation',
    '101_DKA_WeatherStation_Radiation_Global_Tilted',
    '101_DKA_WeatherStation_Weather_Temperature_Celsius'
]
df_weather = df[['timestamp'] + weather_cols].copy()
df_weather.rename(columns={
    '101_DKA_WeatherStation_Global_Horizontal_Radiation': 'ghi',
    '101_DKA_WeatherStation_Radiation_Global_Tilted': 'tilted_radiation',
    '101_DKA_WeatherStation_Weather_Temperature_Celsius': 'temperature'
}, inplace=True)
df_weather['timestamp'] = pd.to_datetime(df_weather['timestamp'], errors='coerce')
for col in ['ghi', 'tilted_radiation', 'temperature']:
    df_weather[col] = pd.to_numeric(df_weather[col], errors='coerce')
df_weather = df_weather.dropna().sort_values('timestamp')

# 4. الدمج الزمني
print("🔗 جاري الدمج...")
df_merged = pd.merge_asof(
    df_power, df_weather, on='timestamp', direction='nearest', tolerance=pd.Timedelta('15min')
)
df_merged = df_merged.dropna()

# 5. الحفظ
clean_path = os.path.join(PROCESSED_DIR, 'forecast_dkasc_clean.csv')
df_merged.to_csv(clean_path, index=False)

print(f"✅ تم التنظيف والدمج! عدد الصفوف: {len(df_merged)}")
print(f"💾 الملف محفوظ في: {clean_path}")

⏳ جاري قراءة DKASC...
🔗 جاري الدمج...


C:\Users\HP\AppData\Local\Temp\ipykernel_16700\625748457.py:44: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  df_power, df_weather, on='timestamp', direction='nearest', tolerance=pd.Timedelta('15min')


✅ تم التنظيف والدمج! عدد الصفوف: 98146
💾 الملف محفوظ في: F:\Abdelrahman1\ME\programming\SIC\Project3.2\data\processed\Forecast\forecast_dkasc_clean.csv


## 2. Kaggle generation and weather merge

Merge Plant 1 and Plant 2 generation data with their weather sensor data, then save `kaggle_merged_clean.csv`.

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "kaggle"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def process_kaggle_plant(plant_num):
    print(f"Preparing plant {plant_num}...")

    gen_path = RAW_DIR / f"Plant_{plant_num}_Generation_Data.csv"
    weather_path = RAW_DIR / f"Plant_{plant_num}_Weather_Sensor_Data.csv"

    df_gen = pd.read_csv(gen_path)
    df_weather = pd.read_csv(weather_path)

    df_gen["DATE_TIME"] = pd.to_datetime(df_gen["DATE_TIME"], format="mixed", dayfirst=True)
    df_weather["DATE_TIME"] = pd.to_datetime(df_weather["DATE_TIME"], format="mixed", dayfirst=True)

    df_merged = pd.merge(
        df_gen,
        df_weather,
        on=["DATE_TIME", "PLANT_ID"],
        how="inner",
        suffixes=("", "_weather"),
    )
    return df_merged


if __name__ == "__main__":
    plant1 = process_kaggle_plant(1)
    plant2 = process_kaggle_plant(2)

    df_kaggle_all = pd.concat([plant1, plant2], ignore_index=True)

    print("Both plants were merged successfully!")
    print(f"Number of rows: {len(df_kaggle_all):,}")
    print(f"Number of columns: {len(df_kaggle_all.columns):,}")

    cols_to_show = [
        "DATE_TIME", "PLANT_ID", "SOURCE_KEY", "AC_POWER",
        "AMBIENT_TEMPERATURE", "IRRADIATION",
    ]
    print("\nData sample:")
    print(df_kaggle_all[cols_to_show].head())

    output_path = PROCESSED_DIR / "kaggle_merged_clean.csv"
    df_kaggle_all.to_csv(output_path, index=False)
    print(f"\nSaved kaggle_merged_clean.csv to: {output_path}")

## 3. Synthetic anomaly dataset

Filter daylight observations, create anomaly labels, inject representative inverter faults, and save `kaggle_anomaly_data.csv`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
input_path = PROCESSED_DIR / "kaggle_merged_clean.csv"
output_path = PROCESSED_DIR / "kaggle_anomaly_data.csv"

df = pd.read_csv(input_path)
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])

print("Filtering daylight observations...")
df_daylight = df[df["IRRADIATION"] > 0.05].copy()
df_daylight["is_anomaly"] = 0

df_daylight = df_daylight.set_index(["DATE_TIME", "SOURCE_KEY"]).sort_index()

inverters = df_daylight.index.get_level_values("SOURCE_KEY").unique()
np.random.seed(42)
selected_inverters = np.random.choice(inverters, size=15, replace=False)

print("Injecting faults into 15 inverters...")
days = df_daylight.index.get_level_values("DATE_TIME").normalize().unique()
selected_days = np.random.choice(days, size=5, replace=False)

for inv in selected_inverters:
    day = np.random.choice(selected_days)

    mask = (
        (df_daylight.index.get_level_values("SOURCE_KEY") == inv)
        & (df_daylight.index.get_level_values("DATE_TIME").normalize() == day)
    )

    if np.random.rand() < 0.33:
        times = df_daylight[mask].index.get_level_values("DATE_TIME")
        if len(times) > 16:
            bad_times = times[len(times) // 3:len(times) // 3 + 16]
            df_daylight.loc[(bad_times, inv), "AC_POWER"] = 0
            df_daylight.loc[(bad_times, inv), "DC_POWER"] = 0
            df_daylight.loc[(bad_times, inv), "is_anomaly"] = 1

    elif np.random.rand() < 0.66:
        times = df_daylight[mask].index.get_level_values("DATE_TIME")
        if len(times) > 24:
            bad_times = times[len(times) // 4:len(times) // 4 + 24]
            df_daylight.loc[(bad_times, inv), "AC_POWER"] *= 0.5
            df_daylight.loc[(bad_times, inv), "is_anomaly"] = 1

    else:
        df_daylight.loc[mask, "AC_POWER"] *= 0.85
        times = df_daylight[mask].index.get_level_values("DATE_TIME")
        if len(times) > 10:
            mid_times = times[len(times) // 2:len(times) // 2 + 10]
            df_daylight.loc[(mid_times, inv), "is_anomaly"] = 1

df_final = df_daylight.reset_index()
df_final = df_final[df_final["AC_POWER"] >= 0]

print("Faults injected successfully!")
print(f"Total rows: {len(df_final):,}")
print(f"Normal rows: {len(df_final[df_final['is_anomaly'] == 0]):,}")
print(f"Anomaly rows: {len(df_final[df_final['is_anomaly'] == 1]):,}")

df_final.to_csv(output_path, index=False)
print(f"Saved kaggle_anomaly_data.csv to: {output_path}")